# WikiLens · dedicated reranker comparison

Use 30 retrieved articles, chunked by headings/subheadings, then choose a ranker:

- `bge_m3`: the previous BGE-M3 dense-similarity method.
- `model2vec`: the requested `Jarbas/m2v-256-bge-reranker-large` static embedder (CPU).
- `cross_encoder`: `BAAI/bge-reranker-v2-m3`, a multilingual query–passage reranker (GPU).

All methods use the same literal source chunks, reference-section filter and answer settings. The final cell shows the saved real comparison without loading models. The original notebooks are preserved.


## Settings

Edit the dictionary below. `answer_target_words` controls article length; `answer_max_new_tokens` budgets the entire JSON response, including supporting quotations. Defaults aim for a 500-word overview with multiple sections, subject to available evidence.

`context_chars` controls retained source text. Actual prompt tokens + requested output must fit `model_context_tokens`. `answer_max_time` gives longer synthesis time to complete. After edits run Settings, Setup, then the following cells. Set the separate review checkpoint to True after reading the evidence.

Prototype mode is the default: `strict_generation=False` generates ordinary article prose without constrained JSON or quotation-rejection checks. Citations are unverified. Turn it on to use the stricter structured path.

`BGE_SETTINGS` controls heading chunk windows. `rerank_top_k`, Qwen relevance threshold and compression budgets are unused here. BGE uses dense cosine similarity, not a cross-encoder; no absolute similarity cutoff is imposed.

Choose the ranking method in `RERANK_SETTINGS`. `search_top_k=30` affects fresh retrieval; the default saved capture already contains 30 articles. Model revisions are pinned in wikilens/reranker_comparison.py. Use scripts/compare-rerankers.py to prepare/download models and execute all three versions.


In [ ]:
# EDIT SETTINGS HERE, then run Setup and the following cells.
# Token limits include JSON and supporting quotations, not just answer prose.
SETTINGS = {
    # Question, personalization and output language
    "question": "winter war",
    "search_focus": "history",  # e.g. "video games", or "" for no focus
    "answer_language": "English",

    # Search input and HyDE (generation budgets include all output tokens)
    "input_mode": "saved",  # "hyde" or "saved" for debugging existing results
    "saved_retrieval_file": "reports/reranker-input-30-20261114.json",
    "hyde_max_new_tokens": 192,  # positive tokens; must leave room for the prompt
    "hyde_max_time": 45,  # seconds, at most 60
    "search_top_k": 30,  # retrieved articles, 1–100
    "search_language": None,  # None = all; or "fi", "en", "sv"

    # Qwen-only controls retained for comparison (unused in BGE postprocessing)
    "rerank_top_k": 10,  # candidate passages Qwen sees; one call per passage
    "min_relevance": 0.5,
    "passage_chars": 1600,
    "passage_overlap": 250,
    "compression_max_new_tokens": 384,  # positive tokens; must leave room for the prompt
    "compression_max_time": 15,  # seconds per call, at most 60
    "postprocessing_seconds": 600,  # total budget, checked between calls

    # Evidence/context budget for the final answer
    "max_evidence_nodes": 10,
    "max_per_article": 3,
    "context_chars": 7000,  # source text only; prompt overhead also uses tokens
    "model_context_tokens": 8192,  # input + requested output, 256–8192
    "embedding_max_batch_tokens": 1024,

    # Final output and citation inspection
    "strict_generation": False,  # False = show prose drafts without JSON/quote rejection
    "answer_target_words": 500,  # prose target, only when supported by evidence
    "answer_max_new_tokens": 4000,  # positive tokens; must leave room for the prompt
    "answer_max_time": 300,  # seconds for article synthesis, at most 600
    "citation_margin_chars": 300,
}

# Review checkpoint only: changing this does NOT require repeating search.
# Set True after inspecting evidence, before running the generation cell.
READY_FOR_GENERATION = False

# Heading/BGE controls: all chunks are scored, not just ten passages.
BGE_SETTINGS = {"chunk_tokens": 384, "overlap_tokens": 48, "max_tokens": 512, "skip_reference_sections": True}

# Choose a ranker; model2vec is a static embedder, cross_encoder scores pairs.
RERANK_SETTINGS = {"method": "cross_encoder", "cross_batch_size": 8}


## 1 · Set up the run

Loads helpers and validates Settings. Re-running this cell releases the previous owned model and starts a fresh report. Use `input_mode="saved"` only to debug saved retrieval, which may not match the current focus.


In [ ]:
from pathlib import Path
import sys
import json
import html
import time
from datetime import datetime, timezone

project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wikilens").is_dir())
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
assert sys.version_info[:2] == (3, 12), "Use Python 3.12."
assert Path(sys.prefix).resolve() == (project_dir / ".venv").resolve(), "Select the project .venv kernel."

import pandas as pd
from IPython.display import HTML, display
import importlib
import wikilens.torch_runtime as runtime_module
runtime_module = importlib.reload(runtime_module)
TorchRuntime = runtime_module.TorchRuntime
import wikilens.output_schemas as schema_module
importlib.reload(schema_module)
import wikilens.node_pipeline as answer_module
importlib.reload(answer_module)
from wikilens.node_pipeline import (
    load_retrieved, generate_answer, citation_context,
)


import wikilens.notebook_settings as settings_module
settings_module = importlib.reload(settings_module)
PipelineSettings = settings_module.PipelineSettings
settings = PipelineSettings.model_validate(SETTINGS)
question = settings.question
answer_language = settings.answer_language
search_focus = settings.search_focus
INPUT_MODE = settings.input_mode
RETRIEVAL_FILE = project_dir / settings.saved_retrieval_file
data_dir = project_dir / "data/s-2026-10"
# Stop an earlier instance if Setup is rerun. No model is loaded by construction.
previous_runtime = globals().get("node_runtime")
if previous_runtime is not None:
    previous_runtime.close()
node_runtime = TorchRuntime(device="cuda", context_tokens=settings.model_context_tokens, max_batch_tokens=settings.embedding_max_batch_tokens)
node_run = {"question": question, "answer_language": answer_language,
            "created_at": datetime.now(timezone.utc).isoformat(),
            "dataset_release": "s-2026-10", "search_focus": search_focus, "input_mode": INPUT_MODE, "settings": settings.model_dump(), "stages": {}, "status": "incomplete"}

def check_settings():
    if PipelineSettings.model_validate(SETTINGS).model_dump() != node_run["settings"]:
        raise ValueError("Settings changed; rerun Setup and subsequent cells.")

def start_node_stage(stage):
    check_settings()
    if (question != node_run["question"] or answer_language != node_run["answer_language"]
        or search_focus != node_run["search_focus"] or INPUT_MODE != node_run["input_mode"]):
        raise ValueError("Question/language/focus/input mode changed; rerun Setup and the following cells.")
    order = ["hyde", "input", "postprocessing", "generation"]
    index = order.index(stage)
    if index and order[index - 1] not in node_run["stages"]:
        raise ValueError("Complete the previous stage first.")
    for name in order[index:]:
        node_run["stages"].pop(name, None)
    node_run["status"] = "incomplete"

print("Question:", question)
print("Search focus:", search_focus, "| input mode:", INPUT_MODE)
print("Saved retrieval (only used in saved mode):", RETRIEVAL_FILE)

node_run["variant"] = "heading_reranker_comparison"
node_run["bge_settings"] = dict(BGE_SETTINGS)

node_run["rerank_settings"] = dict(RERANK_SETTINGS)


## 2 · Generate the search document

Qwen writes a hypothetical document using the topic and focus. Inspect it as a search aid: its statements are not evidence.


In [ ]:
# Reload changed lightweight search code in a long-lived notebook kernel.
import importlib
import wikilens.hyde_search as search_module
search_module = importlib.reload(search_module)
generate_focused_hyde = search_module.generate_focused_hyde
retrieve_hyde = search_module.retrieve_hyde
start_node_stage("hyde")
if INPUT_MODE == "hyde":
    hyde = generate_focused_hyde(node_runtime, question, search_focus=search_focus,
        max_new_tokens=settings.hyde_max_new_tokens, max_time=settings.hyde_max_time)
else:
    hyde = {"status": "skipped", "question": question, "search_focus": search_focus,
            "reason": "Explicit saved-input debugging mode", "seconds": 0.0}
node_run["stages"]["hyde"] = hyde
print("Status:", hyde["status"])
print("HYPOTHETICAL — NOT EVIDENCE:", hyde.get("text"))
if hyde["status"] == "failed": print(hyde["error"])


## 3 · Retrieve real articles

BGE-M3 embeds HyDE; MariaDB retrieves `search_top_k` articles and their original revision metadata. Search settings can be tuned later.


In [ ]:
start_node_stage("input")
if INPUT_MODE == "hyde":
    retrieval = retrieve_hyde(node_runtime, question, node_run["stages"]["hyde"],
                              top_k=settings.search_top_k, language=settings.search_language)
    node_run["stages"]["input"] = retrieval
else:
    saved = json.loads(RETRIEVAL_FILE.read_text())
    if "stages" in saved and "input" in saved["stages"]:
        assert saved["question"] == question, "Saved retrieval question differs; choose the matching topic or use fresh HyDE."
        node_run["stages"]["input"] = {**saved["stages"]["input"], "reused_from": str(RETRIEVAL_FILE), "seconds": 0.0, "original_retrieval_seconds": saved["stages"]["input"].get("seconds")}
    else:
        node_run["stages"]["input"] = {"articles": load_retrieved(RETRIEVAL_FILE, data_dir, branch="hyde"),
            "reused_from": str(RETRIEVAL_FILE), "limitation": "Saved retrieval may not match current focus."}
display(pd.DataFrame([
    {"rank": a.get("retrieval_rank"), "title": a["title"], "lang": a["lang"],
     "revision": a["rev_id"], "characters": len(a["text"])}
    for a in node_run["stages"]["input"]["articles"]
]))


## 4 · Rank identical heading chunks

Sections preserve their exact text, original offsets and source revisions. Long sections use overlapping token windows. Reference sections are excluded. Every method scores all content chunks from the same 30 articles; selection budgets and overlap rules remain fixed.

BGE-M3 and Model2Vec compare independent text embeddings. The cross-encoder reads each question–passage pair together. Their score scales differ and are not calibrated probabilities.

During a timed comparison the chunks are prepared once and replayed from `SHARED_CHUNKS_FILE`. Shared preparation/retrieval time is recorded separately. Standalone notebook runs prepare their own chunks.


In [ ]:
import importlib
import wikilens.reranker_comparison as rerank_module
rerank_module = importlib.reload(rerank_module)
start_node_stage("postprocessing")
chunk_started = time.perf_counter()
if globals().get("SHARED_CHUNKS_FILE"):
    source_chunks = json.loads(Path(SHARED_CHUNKS_FILE).read_text())
    chunk_seconds = 0.0  # prepared once outside the timed method comparisons
else:
    source_chunks = rerank_module.prepare_chunks(node_run["stages"]["input"]["articles"], **BGE_SETTINGS)
    chunk_seconds = time.perf_counter() - chunk_started
postprocessing = rerank_module.rank_chunks(
    node_runtime, question, source_chunks, search_focus=search_focus,
    max_nodes=settings.max_evidence_nodes, context_chars=settings.context_chars,
    max_per_article=settings.max_per_article, max_tokens=BGE_SETTINGS["max_tokens"],
    progress=lambda message: print(message, flush=True), **RERANK_SETTINGS,
)
postprocessing["rank_seconds"] = postprocessing["seconds"]
postprocessing["chunk_seconds"] = chunk_seconds
postprocessing["seconds"] += chunk_seconds
node_run["stages"]["postprocessing"] = postprocessing
print("Ranker:", postprocessing["diagnostics"])
print("Candidates:", postprocessing["candidate_count"], "| selected:", len(postprocessing["nodes"]))
display(pd.DataFrame([
    {"id": n["id"], "title": n["title"], "lang": n["lang"],
     "section": " > ".join(n["heading_path"]), "score": n["rerank_score"], "characters": len(n["text"])}
    for n in postprocessing["nodes"]
]))
for node in postprocessing["nodes"]:
    print(f"[{node['id']}] {' > '.join(node['heading_path'])} ({node['lang']})")
    print(node["text"], "\n", node["revision_url"], "\n")


## 5 · Inspect an original passage (optional)

Choose a selected `CITATION_ID` to reopen its exact source span, surrounding context and revision link.


In [ ]:
if not node_run["stages"]["postprocessing"]["nodes"]:
    print("No selected citation; inspect postprocessing status.")
else:
    CITATION_ID = "S1"
    context = citation_context(
        node_run["stages"]["postprocessing"]["nodes"],
        node_run["stages"]["input"]["articles"],
        CITATION_ID, margin=settings.citation_margin_chars,
    )
    print(context["title"], context["lang"], context["revision_url"])
    print("BEFORE:", context["before"])
    print("SELECTED:", context["passage"])
    print("AFTER:", context["after"])


## 6 · Write the article

Prototype mode (`strict_generation=False`) writes ordinary prose with source IDs. It skips the manual review gate, constrained JSON and quotation rejection. Output is displayed as an **unverified draft**, including text returned by a strict run that failed validation. No second generation call is made automatically.

Set `strict_generation=True` to restore structured decoding and quotation validation. In strict mode set `READY_FOR_GENERATION=True` after reading the evidence. Context-window and GPU limits still apply; generation needs source passages.


In [ ]:
# Upgrade a stale runtime once; keep completed stages and diagnostic records.
import importlib
import wikilens.torch_runtime as runtime_module
runtime_module = importlib.reload(runtime_module)
node_runtime = runtime_module.upgrade_runtime(node_runtime)
import wikilens.output_schemas as schema_module
importlib.reload(schema_module)
import wikilens.node_pipeline as answer_module
answer_module = importlib.reload(answer_module)
generate_answer = answer_module.generate_answer
strict_generation = getattr(settings, "strict_generation", False)
if strict_generation:
    assert READY_FOR_GENERATION, "Read the selected passages before enabling strict generation."
if not node_run["stages"]["postprocessing"].get("nodes"):
    print("No selected evidence: generation will explain that evidence is missing.")
start_node_stage("generation")
answer = generate_answer(
    node_runtime, question, node_run["stages"]["postprocessing"]["nodes"],
    answer_language=answer_language, search_focus=search_focus,
    max_new_tokens=settings.answer_max_new_tokens, max_time=settings.answer_max_time,
    target_words=settings.answer_target_words, strict_validation=strict_generation,
)
node_run["stages"]["generation"] = answer
node_run["status"] = answer["status"]
print("Status:", answer["status"])
if answer["status"] == "draft":
    lookup = {n["id"]: n for n in node_run["stages"]["postprocessing"]["nodes"]}
    paragraphs = []
    evidence_rows = []
    for claim in answer["claims"]:
        links = " ".join(
            '<a href="' + html.escape(lookup[s]["revision_url"], quote=True)
            + '" target="_blank" rel="noopener noreferrer">[' + html.escape(s) + ']</a>'
            for s in dict.fromkeys(claim["sources"])
        )
        if claim.get("heading"):
            paragraphs.append("<h3>" + html.escape(claim["heading"]) + "</h3>")
        paragraphs.append("<p>" + html.escape(claim["text"]) + " <sup>" + links + "</sup></p>")
        for item in claim["evidence"]:
            evidence_rows.append("<p><strong>[" + html.escape(item["id"]) + "]</strong> "
                + html.escape(item["quote"]) + "</p>")
    display(HTML('<article style="max-width:780px;line-height:1.8;font-size:17px;padding:24px 0">'
        + "<h2>" + html.escape(question) + "</h2>"
        + '<p style="font-size:13px;opacity:.65">AI draft · citations checked for literal source provenance · review factual support</p>'
        + "".join(paragraphs) + "</article>"
        + "<details><summary>Supporting quotations</summary>" + "".join(evidence_rows) + "</details>"))
    print("Article words:", sum(len(c["text"].split()) for c in answer["claims"]))
elif answer["status"] == "draft_unverified":
    # Escape all model content; link only IDs that exist in this run.
    import re
    lookup = {n["id"]: n for n in node_run["stages"]["postprocessing"]["nodes"]}
    body = []
    for block in answer["draft_text"].split("\n\n"):
        escaped = html.escape(block.strip())
        def source_link(match):
            source = lookup.get(match.group(1))
            if source is None:
                return match.group(0) + " (unknown source)"
            return ('<a title="Source link only; support not verified" href="'
                    + html.escape(source["revision_url"], quote=True)
                    + '" target="_blank" rel="noopener noreferrer">' + match.group(0) + '</a>')
        escaped = re.sub(r"\[(S\d+)\]", source_link, escaped)
        if escaped.startswith("#") and "\n" not in escaped:
            body.append("<h3>" + escaped.lstrip("# ") + "</h3>")
        else:
            body.append('<p style="white-space:pre-wrap">' + escaped + "</p>")
    display(HTML('<article style="max-width:780px;line-height:1.8;font-size:17px;padding:24px 0">'
        + "<h2>" + html.escape(question) + "</h2>"
        + '<p style="font-size:13px;color:#8a5600">Prototype draft · citations and factual support unverified</p>'
        + "".join(body) + "</article>"))
    if answer.get("error"):
        print("Validation note (draft retained):", answer["error"])
else:
    print(answer.get("error", "No answer was generated from the available evidence."))
    if answer.get("raw_output"):
        print(answer["raw_output"])


## 7 · Save and release GPU memory

Run even after a failed stage. Saves settings, prompts, HyDE, retrieval, compression judgments, article, citations and runtime diagnostics. Download reports before the temporary remote job ends. Nothing is marked human-reviewed automatically.


In [ ]:
check_settings()
assert question == node_run["question"] and answer_language == node_run["answer_language"] and search_focus == node_run["search_focus"] and INPUT_MODE == node_run["input_mode"], "Rerun Setup after changing question/language."
node_runtime.close()
node_run["runtime"] = node_runtime.stats()
node_run["seconds"] = sum(s.get("seconds", 0.0) for s in node_run["stages"].values())
node_run["timing_basis"] = "Sum of measured stages, excluding interactive pauses and setup."
node_run["claim_support_reviewed"] = False
report_dir = project_dir / "reports"
report_dir.mkdir(exist_ok=True)
node_report_path = report_dir / ("reranker-answer-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
node_report_path.write_text(json.dumps(node_run, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", node_report_path)
print("Owned GPU model released.")


## 8 · Read a full source article

Change `SOURCE_INDEX` below to read any retrieved article. This shows the full stored source text before compression, in its original language, with its exact revision link. No model call is needed.


In [ ]:
# Change this index to inspect another retrieved source (0 = first article).
SOURCE_INDEX = 0

from IPython.display import HTML, display
import html

source_articles = node_run.get("stages", {}).get("input", {}).get("articles", [])
if not source_articles:
    print("Run retrieval first to load source articles.")
else:
    print("Available sources:")
    for index, article in enumerate(source_articles):
        print(f"{index}: {article['title']} ({article['lang']})")
    if not 0 <= SOURCE_INDEX < len(source_articles):
        print(f"Choose SOURCE_INDEX between 0 and {len(source_articles) - 1}.")
    else:
        source_article = source_articles[SOURCE_INDEX]
        source_url = (f"https://{source_article['lang']}.wikipedia.org/w/index.php"
                      f"?oldid={source_article['rev_id']}")
        display(HTML(
            '<section style="max-width:900px;line-height:1.7;padding:20px 0">'
            + '<h2>' + html.escape(source_article['title']) + '</h2>'
            + '<p>' + html.escape(source_article['lang'].upper())
            + ' · Revision ' + html.escape(str(source_article['rev_id']))
            + ' · <a href="' + html.escape(source_url, quote=True)
            + '" target="_blank" rel="noopener noreferrer">Open original revision</a></p>'
            + '<div style="white-space:pre-wrap;overflow-wrap:anywhere">'
            + html.escape(source_article['text']) + '</div></section>'
        ))


## Saved comparison · rankings, times and full drafts

Run this cell on its own to see the actual three-way comparison. Source IDs belong to their own variant. Outputs are unverified drafts; model scores do not establish factual support.


In [ ]:
from pathlib import Path
import json
import html
import pandas as pd
from IPython.display import display
project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wikilens").is_dir())
from IPython.display import Markdown

comparison_file = project_dir / "reports/reranker-comparison-latest.json"
if not comparison_file.exists():
    print("Run scripts/compare-rerankers.py first to produce the real comparison.")
else:
    comparison = json.loads(comparison_file.read_text())
    print("Measured topic:", comparison["question"])
    print(comparison["timing_basis"])
    display(pd.DataFrame([
        {"variant": name, "candidates": item["run"]["stages"]["postprocessing"]["candidate_count"],
         "selected": len(item["run"]["stages"]["postprocessing"]["nodes"]),
         "postprocessing_s": round(item["metrics"]["postprocessing_seconds"], 2),
         "generation_s": round(item["metrics"]["generation_seconds"], 2),
         "combined_s": round(item["metrics"]["combined_seconds"], 2),
         "output_words": item["metrics"]["output_words"]}
        for name, item in comparison["variants"].items()
    ]))
    for name, item in comparison["variants"].items():
        generation = item["run"]["stages"]["generation"]
        text = generation.get("draft_text") or generation.get("raw_output") or generation.get("error", "No output")
        display(Markdown("### " + name + "\n\n*AI draft — factual support unverified.*\n\n" + html.escape(text)))
